In [4]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader,random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models
import pandas as pd
import numpy as np
import PIL
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image



In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # resize all images to 224x224
    transforms.ToTensor(),          # convert image to tensor [0,1]
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])  # normalize to [-1,1]
])

images_root = "caltech101"   # <- change to the best path you found above
dataset = ImageFolder(images_root, transform=transform)

train_size=int(0.8*len(dataset))
test_size=int(len(dataset)-train_size)
train_dataset,test_dataset=random_split(dataset,[train_size,test_size])

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,num_workers=2)

In [6]:
model=models.resnet34(pretrained=True)

for p in model.parameters():
    p.requires_grad = False

for p in model.layer4.parameters():
    p.requires_grad = True
for p in model.fc.parameters():
    p.requires_grad = True

num_features=model.fc.in_features
model.fc=nn.Linear(num_features,len(dataset.classes))
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Move the model to the device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)


c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
num_epochs = 10

# Train the model
for epoch in range(num_epochs):
    # Train the model on the training set
    model.train()
    train_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        # Move the data to the device
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Update the training loss
        train_loss += loss.item() * inputs.size(0)



FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\_utils\worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataset.py", line 416, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
            ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\folder.py", line 245, in __getitem__
 

In [ ]:
    # Evaluate the model on the test set
model.eval()
for epoch in range(num_epochs):

    test_loss = 0.0
    test_acc = 0.0
    with torch.no_grad():
        for i, (inputs, labels) in enumerate(test_loader):
            # Move the data to the device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Update the test loss and accuracy
            test_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            test_acc += torch.sum(preds == labels.data)

    # Print the training and test loss and accuracy
    test_loss /= len(test_dataset)
    test_acc = test_acc.double() / len(test_dataset)
    print(f"Epoch [{epoch + 1}/{num_epochs}] Test Loss: {test_loss:.4f} Test Acc: {test_acc:.4f}")

Epoch [1/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [2/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [3/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [4/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [5/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [6/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [7/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [8/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [9/10] Test Loss: 0.1886 Test Acc: 0.9551
Epoch [10/10] Test Loss: 0.1886 Test Acc: 0.9551


In [ ]:
import torch
import torch.nn.functional as F

# FGSM attack builder
def fgsm_attack(model, images, labels, epsilon,
                device,
                mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5),
                loss_fn=torch.nn.CrossEntropyLoss()):
    
    # Ensure copies and device
    images = images.clone().detach().to(device)
    labels = labels.to(device)
    model.eval()   # set eval to avoid e.g. dropout randomness while crafting

    # make inputs require gradients
    images.requires_grad = True

    # forward
    outputs = model(images)
    loss = loss_fn(outputs, labels)

    # backward on input
    model.zero_grad()
    loss.backward()

    # gradient of loss w.r.t. normalized images
    grad = images.grad.data  # shape (N,C,H,W)

    # convert epsilon from pixel space to normalized space per channel:
    # normalized = (pixel - mean)/std -> pixel = normalized*std + mean
    # so a delta in pixel space corresponds to delta_norm = delta_pixel / std
    std_t = torch.tensor(std, device=device).view(1, -1, 1, 1)
    eps_norm = epsilon / std_t  # broadcastable to (N,C,H,W)

    # FGSM perturbation in normalized space (untargeted): add sign(grad)
    perturbed = images + eps_norm * grad.sign()

    # clamp to valid normalized range so pixel values remain in [0,1]
    mean_t = torch.tensor(mean, device=device).view(1, -1, 1, 1)
    min_norm = (0.0 - mean_t) / std_t
    max_norm = (1.0 - mean_t) / std_t
    perturbed = torch.max(torch.min(perturbed, max_norm), min_norm)

    return perturbed.detach()


In [ ]:


def gradcam_for_resnet(model, img_tensor, device='cpu'):
    """
    model: ResNet model
    img_tensor: torch tensor (C,H,W) normalized (single image)
    target_class: int or None (if None uses model prediction)
    returns: (orig_image_HWC, cam_gray, overlay_image)
    """
    # prepare
    model.to(device).eval()
    input_tensor = img_tensor.unsqueeze(0).to(device)   # (1,C,H,W)

    # get prediction if target_class not provided
    with torch.no_grad():
        logits = model(input_tensor)
    pred = int(logits.argmax(dim=1).item())
    target = pred 

    # Grad-CAM: use last conv layer of ResNet (layer4[-1])
    target_layers = [model.layer4[-1]]
    cam = GradCAM(model=model, target_layers=target_layers)

    # compute cam (returns numpy origay (batch, H, W))
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(target)])
    cam_map = grayscale_cam[0]  # HxW, values [0..1]


    return  cam_map
  
  

    

In [ ]:
def process_original(img,mean,std):
   orig_disp = img.squeeze(0).cpu().numpy()  # (C,H,W) or (1,H,W)
   orig_disp = np.transpose(orig_disp, (1, 2, 0))
            
   mean = np.array(mean).reshape(1,1,3)
   std  = np.array(std).reshape(1,1,3)
   orig_disp = (orig_disp * std) + mean
   return np.clip(orig_disp, 0, 1)

In [ ]:
import random
import torch


model.to(device)
model.eval()
epsilons = [0 ,0.001,0.01,0.1,0.2,0.3]  

N_images = 5
mean=[0.5,0.5,0.5]
std=[0.5,0.5,0.5]

test_samples = []
for inputs, labels in test_loader:
    for i in range(len(inputs)):
        test_samples.append((inputs[i], labels[i]))

chosen_samples = random.sample(test_samples, N_images)

for e in epsilons:
  print(f"For epsilon with value {e}")
  
  for idx, (img, label) in enumerate(chosen_samples):
    img = img.unsqueeze(0).to(device)   
    label = label.unsqueeze(0).to(device)

    orig_img = img.clone().detach()

    img = fgsm_attack(model, img, label, e, device)

    img = img.clone().detach().requires_grad_(True)

  
    model.zero_grad()
    if img.grad is not None:
        img.grad.zero_()


    pred = model(img)
    class_idx = pred.argmax(dim=1)
  
    print(f"Image {idx+1}:")
    print(f"  True label       : {label.item()}")
    print(f"   prediction   : {class_idx.item()}")
    print("-" * 40)
    
    class_neuron=pred[0,class_idx.item()]
    class_neuron.backward()

    dimg=img.grad.squeeze(0).cpu().detach().numpy()
    saliency = np.max(np.abs(dimg), axis=0) 

    sal_min, sal_max = saliency.min(), saliency.max()
    saliency_norm = (saliency - sal_min) / (sal_max - sal_min + 1e-8)
    
    orig_disp=process_original(orig_img,mean,std)
    

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    ax = axes[0]
    if orig_disp.ndim == 3:
      ax.imshow(orig_disp)
    
    ax = axes[1]
    im = ax.imshow(saliency_norm, cmap='hot')
    ax.set_title(f"Saliency (max over channels)\nEps {e}")
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = axes[2]
    if orig_disp.ndim == 3:
        ax.imshow(orig_disp)
        ax.imshow(saliency_norm, cmap='jet', alpha=0.4)
    
    ax.set_title("Overlay (saliency on image)")
    ax.axis('off')

    plt.tight_layout()
    plt.show()

    target_layers = [model.layer4[-1]]
    target= int(class_idx.item())
    input_tensor = img.to(device)

    cam = GradCAM(model=model, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(target)])
    cam_map = grayscale_cam[0]  

    overlay = show_cam_on_image(orig_disp, cam_map, use_rgb=True)  # returns uint8 HxWx3

    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(cam_map, cmap='hot'); plt.title("Grad-CAM"); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(overlay); plt.title("Overlay"); plt.axis('off')
    plt.tight_layout()
    plt.show()

    print("orig min/max:", orig_disp.min(), orig_disp.max())
    print("cam_map min/max:", cam_map.min(), cam_map.max())


In [ ]:
def pgd_attack(model, x, y, epsilon, alpha, iters, loss_fn, random_start=True):
    """
    Projected Gradient Descent (stronger multi-step attack).
    epsilon : max perturbation (L_inf) in normalized space
    alpha : step size
    iters : number of iterations
    """
    if random_start:
        x_adv = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    else:
        x_adv = x.clone().detach()
    x_adv = torch.max(torch.min(x_adv, clamp_max), clamp_min).detach()
    x_adv.requires_grad_(True)
    for i in range(iters):
        logits = model(x_adv)
        loss = loss_fn(logits, y)
        model.zero_grad()
        loss.backward()
        grad = x_adv.grad.data.sign()
        x_adv = x_adv + alpha * grad
        # projection: keep within epsilon-ball of original x and within valid range
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)
        x_adv = torch.max(torch.min(x_adv, clamp_max), clamp_min)
        x_adv = x_adv.detach()
        x_adv.requires_grad_(True)
    return x_adv.detach()

In [ ]:
import copy

num_epochs = 10
epsilon = 8/255   
mean = (0.5, 0.5, 0.5)
std  = (0.5, 0.5, 0.5)
loss_fn_for_attack = torch.nn.CrossEntropyLoss()

if "model_copy" not in locals():
    model_copy = copy.deepcopy(model).to(device)


# --- Split the existing train_dataset into train/val subsets (not dataset) ---
val_frac = 0.2
train_size = int((1.0 - val_frac) * len(train_dataset))
val_size = len(train_dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size],
                                                         generator=generator)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_subset,   batch_size=32, shuffle=False, num_workers=2)

for epoch in range(num_epochs):
    model_copy.train()
    train_loss = 0.0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        batch_size = inputs.size(0)
        perm = torch.randperm(batch_size, device=device)
        split = batch_size // 2
        idx_adv = perm[:split]         # will be attacked
        idx_clean = perm[split:]       # will remain clean

        inputs_adv = inputs[idx_adv] if idx_adv.numel() > 0 else None
        labels_adv = labels[idx_adv] if idx_adv.numel() > 0 else None
        inputs_clean = inputs[idx_clean] if idx_clean.numel() > 0 else None
        labels_clean = labels[idx_clean] if idx_clean.numel() > 0 else None

        # craft adversarial examples only for the adv half (fgsm_attack sets model_copy.eval() internally)
        if inputs_adv is not None:
            adv_inp_half = fgsm_attack(
                model_copy, inputs_adv, labels_adv, epsilon,
                device,
                mean=mean, std=std,
                loss_fn=loss_fn_for_attack
            )
            # fgsm_attack set model_copy.eval(); return to train mode
            model_copy.train()
        else:
            adv_inp_half = None

        # Zero gradients and do a single forward/backward on mixed data
        optimizer.zero_grad()

        loss = 0.0
        if inputs_clean is not None and inputs_clean.size(0) > 0:
            outputs_clean = model_copy(inputs_clean)
            loss_clean = criterion(outputs_clean, labels_clean)
            loss += (inputs_clean.size(0) / batch_size) * loss_clean
        else:
            outputs_clean = None

        if adv_inp_half is not None and adv_inp_half.size(0) > 0:
            outputs_adv = model_copy(adv_inp_half)
            loss_adv = criterion(outputs_adv, labels_adv)
            loss += (adv_inp_half.size(0) / batch_size) * loss_adv
        else:
            outputs_adv = None

        # Backprop once and step
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_size

    # average training loss over all training samples
    train_loss = train_loss / len(train_subset)

    # ---------- Validation ----------
    model_copy.eval()
    val_loss = 0.0
    val_correct = 0

    for i, (inputs, labels) in enumerate(val_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        batch_size = inputs.size(0)
        perm = torch.randperm(batch_size, device=device)
        split = batch_size // 2
        idx_adv = perm[:split]
        idx_clean = perm[split:]

        inputs_adv = inputs[idx_adv] if idx_adv.numel() > 0 else None
        labels_adv = labels[idx_adv] if idx_adv.numel() > 0 else None
        inputs_clean = inputs[idx_clean] if idx_clean.numel() > 0 else None
        labels_clean = labels[idx_clean] if idx_clean.numel() > 0 else None

        # craft adversarial half (needs grads) BEFORE turning off grads
        if inputs_adv is not None:
            adv_inp_half = fgsm_attack(
                model_copy, inputs_adv, labels_adv, epsilon,
                device,
                mean=mean, std=std,
                loss_fn=loss_fn_for_attack
            )
            # keep model_copy in eval for evaluation
        else:
            adv_inp_half = None

        with torch.no_grad():
            batch_loss = 0.0

            if inputs_clean is not None and inputs_clean.size(0) > 0:
                outputs_clean = model_copy(inputs_clean)
                loss_clean = criterion(outputs_clean, labels_clean)
                batch_loss += (inputs_clean.size(0) / batch_size) * loss_clean
                _, preds_clean = torch.max(outputs_clean, 1)
                val_correct += torch.sum(preds_clean == labels_clean).item()
            else:
                outputs_clean = None

            if adv_inp_half is not None and adv_inp_half.size(0) > 0:
                outputs_adv = model_copy(adv_inp_half)
                loss_adv = criterion(outputs_adv, labels_adv)
                batch_loss += (adv_inp_half.size(0) / batch_size) * loss_adv
                _, preds_adv = torch.max(outputs_adv, 1)
                val_correct += torch.sum(preds_adv == labels_adv).item()
            else:
                outputs_adv = None

        val_loss += batch_loss.item() * batch_size

    # average validation loss and accuracy over val set
    val_loss = val_loss / len(val_subset)
    val_acc = val_correct / len(val_subset)

    print(f"Epoch [{epoch + 1}/{num_epochs}] train Loss: {train_loss:.4f} val Loss: {val_loss:.4f} val Acc: {val_acc:.4f}")


Epoch [1/10] train Loss: 1.3217 val Loss: 1.2033 val Acc: 0.7135
Epoch [2/10] train Loss: 1.3214 val Loss: 1.2345 val Acc: 0.6991
Epoch [3/10] train Loss: 1.2926 val Loss: 1.3145 val Acc: 0.6955
Epoch [4/10] train Loss: 1.3269 val Loss: 1.2899 val Acc: 0.6940
Epoch [5/10] train Loss: 1.3013 val Loss: 1.2466 val Acc: 0.7041
Epoch [6/10] train Loss: 1.3196 val Loss: 1.3230 val Acc: 0.6904
Epoch [7/10] train Loss: 1.3244 val Loss: 1.2777 val Acc: 0.7070


RuntimeError: [enforce fail at alloc_cpu.cpp:121] data. DefaultCPUAllocator: not enough memory: you tried to allocate 4718592 bytes.

In [ ]:
def evaluate_clean_and_adv(m, dataloader, device, epsilon, mean, std, loss_fn_for_attack):
    m.eval()
    total_clean, correct_clean, loss_sum_clean = 0, 0, 0.0
    total_adv, correct_adv, loss_sum_adv = 0, 0, 0.0

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # ----- clean -----
        with torch.no_grad():
            outputs_clean = m(inputs)
            loss_clean = criterion(outputs_clean, labels)
            loss_sum_clean += loss_clean.item() * inputs.size(0)
            _, preds_clean = torch.max(outputs_clean, 1)
            correct_clean += torch.sum(preds_clean == labels).item()
            total_clean += inputs.size(0)

        # ----- adversarial -----
        adv_inputs = fgsm_attack(
            m, inputs.clone().detach(), labels, epsilon,
            device,
            mean=mean, std=std,
            loss_fn=loss_fn_for_attack
        )

        with torch.no_grad():
            outputs_adv = m(adv_inputs)
            loss_adv = criterion(outputs_adv, labels)
            loss_sum_adv += loss_adv.item() * inputs.size(0)
            _, preds_adv = torch.max(outputs_adv, 1)
            correct_adv += torch.sum(preds_adv == labels).item()
            total_adv += inputs.size(0)

    return {
        "clean_loss": loss_sum_clean / total_clean,
        "clean_acc": correct_clean / total_clean,
        "adv_loss": loss_sum_adv / total_adv,
        "adv_acc": correct_adv / total_adv,
    }

results_orig = evaluate_clean_and_adv(model, test_loader, device, epsilon, mean, std, loss_fn_for_attack)
results_copy = evaluate_clean_and_adv(model_copy, test_loader, device, epsilon, mean, std, loss_fn_for_attack)

print("Original model:")
print("   Clean  - loss: {:.4f}, acc: {:.4f}".format(results_orig["clean_loss"], results_orig["clean_acc"]))
print("   Attack - loss: {:.4f}, acc: {:.4f}".format(results_orig["adv_loss"], results_orig["adv_acc"]))

print("\nAdversarially trained model:")
print("   Clean  - loss: {:.4f}, acc: {:.4f}".format(results_copy["clean_loss"], results_copy["clean_acc"]))
print("   Attack - loss: {:.4f}, acc: {:.4f}".format(results_copy["adv_loss"], results_copy["adv_acc"]))



NameError: name 'model' is not defined